# Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

- 🤝 Breakout Room #1
  1. Task 1: Installing Required Libraries
  2. Task 2: Set Environment Variables
  3. Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  4. Task 4: Evaluating our Pipeline with Ragas
  5. Task 6: Making Adjustments and Re-Evaluating

But first! Let's set some dependencies!

## Dependencies and API Keys:

We'll also need to provide our API keys.

First, OpenAI's for our LLM/embedding model combination!

In [1]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/owendewing/Owen-AI7/08_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/owendewing/Owen-AI7/08_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/owendewing/Owen-AI7/08_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'afe4c1'. Skipping!
Property 'summary' already exists in node 'fe5af0'. Skipping!
Property 'summary' already exists in node 'd88cef'. Skipping!
Property 'summary' already exists in node '786aac'. Skipping!
Property 'summary' already exists in node 'd97a0c'. Skipping!
Property 'summary' already exists in node '7335dc'. Skipping!
Property 'summary' already exists in node '693bc8'. Skipping!
Property 'summary' already exists in node '463c13'. Skipping!
Property 'summary' already exists in node '4e4dec'. Skipping!
Property 'summary' already exists in node '5a17ae'. Skipping!
Property 'summary' already exists in node '3c340d'. Skipping!
Property 'summary' already exists in node 'b258fc'. Skipping!
Property 'summary' already exists in node '09981a'. Skipping!
Property 'summary' already exists in node 'dc82ad'. Skipping!


...

Property 'summary_embedding' already exists in node '09981a'. Skipping!
Property 'summary_embedding' already exists in node '693bc8'. Skipping!
Property 'summary_embedding' already exists in node '7335dc'. Skipping!
Property 'summary_embedding' already exists in node '3c340d'. Skipping!
Property 'summary_embedding' already exists in node 'dc82ad'. Skipping!
Property 'summary_embedding' already exists in node 'd97a0c'. Skipping!
Property 'summary_embedding' already exists in node 'b258fc'. Skipping!
Property 'summary_embedding' already exists in node 'fe5af0'. Skipping!
Property 'summary_embedding' already exists in node 'afe4c1'. Skipping!
Property 'summary_embedding' already exists in node '4e4dec'. Skipping!
Property 'summary_embedding' already exists in node '5a17ae'. Skipping!
Property 'summary_embedding' already exists in node '463c13'. Skipping!
Property 'summary_embedding' already exists in node '786aac'. Skipping!
Property 'summary_embedding' already exists in node 'd88cef'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Whaat is the impackt of diffrent academic cale...,"[non-term (includes clock-hour calendars), or ...","Whether an academic calendar is standard term,...",single_hop_specifc_query_synthesizer
1,Whaat are the requirments for includin phsyica...,[Inclusion of Clinical Work in a Standard Term...,Clinical work in physical therapy may be inclu...,single_hop_specifc_query_synthesizer
2,"How Title IV work for payment periods, it got ...",[Non-Term Characteristics A program that measu...,Payment period is for all Title IV programs ex...,single_hop_specifc_query_synthesizer
3,where i find Appendix A?,[both the credit or clock hours and the weeks ...,Appendix A is at the end of this chapter.,single_hop_specifc_query_synthesizer
4,Under what conditions can practicum or clinica...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Practicum or clinical experiences required for...,multi_hop_abstract_query_synthesizer
5,how does accelerated student progression in se...,[<1-hop>\n\nboth the credit or clock hours and...,when a student accelerates their progression i...,multi_hop_abstract_query_synthesizer
6,Under what conditions can practicum or clinica...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Practicum or clinical experiences required for...,multi_hop_abstract_query_synthesizer
7,If a medical or education program requires a p...,[<1-hop>\n\nInclusion of Clinical Work in a St...,When a medical or education program requires a...,multi_hop_abstract_query_synthesizer
8,how direct loan program work for student in no...,[<1-hop>\n\nnon-term (includes clock-hour cale...,for student in non-term or subscription-based ...,multi_hop_specific_query_synthesizer
9,"Acccording to Volume 2, Chapter 2 and Volume 8...",[<1-hop>\n\nnon-term (includes clock-hour cale...,"In subscription-based programs, as detailed in...",multi_hop_specific_query_synthesizer


## LangChain RAG

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [6]:
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

1102

#### ❓ Question: 

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### ✅ Answer:

Chunk_overlap is a helpful tool that ensures that relevant information and specific ideas don't get split between different chunks. Thus, this means that chunks will share some content, helping preserve context across all chunks.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="loan_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="loan_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [10]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

Now we can produce a node for retrieval!

In [12]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### Augmented

Let's create a simple RAG prompt!

In [13]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [15]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [16]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [17]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [18]:
response = graph.invoke({"question" : "What are the different kinds of loans?"})

In [19]:
response["response"]

'Based on the provided context, the document primarily discusses different types of loans related to student financial aid, specifically focusing on Direct Loans. The key types of loans mentioned are:\n\n1. **Direct Loans**: These are loans originated through the Department of Education, including:\n   - **Direct Unsubsidized Loans**: Loans where interest accrues during periods when the borrower is in school, and the borrower has the option to pay the interest while in school.\n   - (Implied but not explicitly named in the context) **Direct Subsidized Loans**: Loans where the government pays the interest during certain periods, though not explicitly detailed in the provided text.\n\nThe context also refers to various loan-related concepts such as loan limits, repayment plans, loan consolidation, and loan counseling, but the primary distinct types of loans explicitly discussed are the different forms of Direct Loans.\n\n**Summary:**\n- Direct Loans (including Direct Unsubsidized Loans a

## Evaluating the App with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [20]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [22]:
dataset.samples[0].eval_sample.response

'The impact of different academic calendars on Title IV aid disbursement includes several key considerations:\n\n1. **Timing of Disbursements:** The timing of when aid is disbursed is affected by whether the academic calendar is standard term, nonstandard term, non-term, or subscription-based. For example, in term-based programs using standard terms (semesters, trimesters, quarters), aid is typically disbursed during the specific payment periods aligned with those terms.\n\n2. **Disbursement Timing Rules:** For Pell Grant, TEACH Grant, and FSEOG programs, disbursements are generally made during the academic term for credit-hour programs using standard or substantially equal nonstandard terms. However, for the Direct Loan Program, disbursement timing depends on whether the program uses standard or nonstandard terms and whether those terms are substantially equal in length. Nonstandard or non-term-based calendars require different rules for timing.\n\n3. **Academic Year and Progression:*

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [23]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [24]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [25]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[53]: TimeoutError()
Exception raised in Job[71]: TimeoutError()


{'context_recall': 0.7542, 'faithfulness': 0.9048, 'factual_correctness': 0.6783, 'answer_relevancy': 0.8748, 'context_entity_recall': 0.2534, 'noise_sensitivity_relevant': 0.2662}

## Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!

> NOTE: This will be using Cohere's Rerank model - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

In [26]:
os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")


We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [27]:
adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [28]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [29]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [30]:
response = adjusted_graph.invoke({"question" : "What are the different kinds of loans?"})
response["response"]

'The provided context mentions different loan types, including:\n\n- Federal PLUS Loans\n- Federal Family Education Loan (FFEL) Program loans (before July 1, 2010)\n- Direct Subsidized Loans\n- Direct Unsubsidized Loans\n- Student Direct PLUS Loans\n\nThese are the types of loans referenced in the context.'

In [31]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [32]:
rerank_dataset.samples[0].eval_sample.response

'The impact of different academic calendars on Title IV aid disbursement is that the type of calendar—whether it is a standard term, nonstandard term, non-term, or subscription-based—affects how aid is awarded and disbursed. For example, schools using a standard term calendar (such as semesters, trimesters, or quarters) have specific procedures for counting instructional weeks and scheduling aid payments. The academic calendar determines the timing and method of disbursement, ensuring aid aligns with instructional periods and student enrollment.'

In [33]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [34]:
result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

{'context_recall': 0.7361, 'faithfulness': 0.7979, 'factual_correctness': 0.5550, 'answer_relevancy': 0.9453, 'context_entity_recall': 0.3175, 'noise_sensitivity_relevant': 0.2653}

#### ❓ Question: 

Which system performed better, on what metrics, and why?

##### ✅ Answer:

| Metric                  | Original Run                       | Reranked Run                             |
| ----------------------- | ---------------------------------- | ---------------------------------------- |
| **LLMContextRecall**    | 0.75                               | 0.74                                     |
| **Faithfulness**        | 0.90                               | 0.80                                     |
| **FactualCorrectness**  | 0.68                               | 0.56                                     |
| **ResponseRelevancy**   | 0.87                               | 0.95                                     |
| **ContextEntityRecall** | 0.25                               | 0.32                                     |
| **NoiseSensitivity**    | 0.27                               | 0.27                                     |

- The LLM Context Recall metric was pretty constant between the two; however, I'm suprised that it didn't go up for the Reranked Run, because the reranked model is supposed to reorder documents by relevancy to the query. I would've thought that this would help the LLM ground its answers in the context.

- The Faithfulness was higher on the original run, which I also am suprised about. Faithfulness measures if the response is supported by the context, and I thought that the reranking would have more faithfulness beacuse it is supposed to select the top 5 most relevant documents. Maybe the reranking process filtered out documents that had small but important details.

- The Factual Correctness also went down, which again I am suprised about. Like I mentioned earlier, maybe during the reranking process, important factual details were dropped. Also, maybe it would improve if we increased k to 10 instead of 5, allowing the model to retain more useful context.

- The Response Relevancy went up a little bit, which makes sense. Likely, the rerank model picked snippets with more keyword overlap and matching phrases/ideas, increasing the relevancy.

- The Context Entity Recall also went up, and I think this is similarly because the reranked context contained more ideas and entities that were reflected in the answer.

- Finally, the noise sensitivity remained the same between both runs, meaning that neither run was better at ignoring irrelevant context.